In [2]:
import requests  #send HTTP requests
from bs4 import BeautifulSoup   #parse html
import json   #manipulate JSON data
import sqlite3   #create database
import re   #regex expressions
import time
import pandas as pd

# 1. Extract

In [3]:
#download webpage

url="https://www.aym-studio.com/products/astoria-maxi-dress"   #webpage url
response=requests.get(url)  #send HTTP request
html=response.text   #store the server's reply 




## test
print(len(html))

802896


In [4]:
#parse HTML document

soup=BeautifulSoup(html, "html.parser")  #convert raw HTML into a navigable parse tree



## test
print(soup.title)

<title>
      Astoria Maxi Dress
    </title>


- ### **Structured JSON:**

In [5]:
#extract JSON data 

script_tag=soup.find("script", type="application/ld+json")    #find the JSON block 
json_data=json.loads(script_tag.string)   #convert it to Python dict

In [6]:
#inspect JSON structure

print("Top-level keys:")
print(json_data.keys())

print("\nVariant keys:")
print(json_data["hasVariant"][0].keys())

print("\nOffer structure:")
print(json_data["hasVariant"][0]["offers"])

Top-level keys:
dict_keys(['@context', '@id', '@type', 'brand', 'category', 'description', 'hasVariant', 'name', 'productGroupID', 'url'])

Variant keys:
dict_keys(['@id', '@type', 'gtin', 'image', 'name', 'offers', 'sku'])

Offer structure:
{'@id': '/products/astoria-maxi-dress?variant=55275849646457#offer', '@type': 'Offer', 'availability': 'http://schema.org/InStock', 'price': '179.00', 'priceCurrency': 'GBP', 'url': 'https://www.aym-studio.com/products/astoria-maxi-dress?variant=55275849646457'}


In [7]:
#extract product metadata from JSON

brand=json_data["brand"]["name"]      
category=json_data["category"]        
product_name=json_data["name"]        
price=json_data["hasVariant"][0]["offers"]["price"]  
currency=json_data["hasVariant"][0]["offers"]["priceCurrency"] 
product_url=json_data["hasVariant"][0]["offers"]["url"]  

## test
print(brand)
print(category)
print(product_name)
print(price, currency)
print(product_url)

AYM
Dresses
Astoria Maxi Dress
179.00 GBP
https://www.aym-studio.com/products/astoria-maxi-dress?variant=55275849646457


- ### **Unstructured HTML:**

In [8]:
#extract materials from HTML
#materials are not available in the structured JSON data
#through manual inspection of the HTML (using browser developer tools),
#the material composition was located inside a <span> element with class:
#"metafield-multi_line_text_field"
#this class appears to be generated by Shopify metafields and is used to store
#product-specific attributes such as fabric composition

material_blocks=soup.find_all(
    "span",
    class_="metafield-multi_line_text_field"
)

#select the block containing a percentage symbol indicative of materials info
for block in material_blocks:
    text=block.get_text(separator=" ", strip=True)
    if "%" in text:
        materials_raw=text
        break

## test
if materials_raw:
    print(materials_raw)
else:
    print('Materials not found')

68% Bamboo Viscose, 28% Cotton, 4% Elastane. Fabric country of origin: Turkey


# 2. Transform

In [9]:
#clean material composition 

def clean_materials(text):
    #keep only the first sentence (before extra details)
    return text.split(".")[0]

materials_clean=clean_materials(materials_raw)



## test
print(materials_clean)

68% Bamboo Viscose, 28% Cotton, 4% Elastane


# 3. Load

In [16]:
#create/ connect databse 

conn=sqlite3.connect("prototype.db")
cursor=conn.cursor()

In [12]:
#create products table 

cursor.execute("""
CREATE TABLE products (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    brand TEXT,
    category TEXT,
    price REAL,
    currency TEXT,
    materials TEXT
)
""")

In [13]:
#insert product into database

cursor.execute("""
INSERT INTO products (brand, category, price, currency, materials)
VALUES (?, ?, ?, ?, ?)
""", (brand, category, price, currency, materials_clean))

conn.commit()

In [14]:
## test
cursor.execute("SELECT * FROM products")
rows=cursor.fetchall()
for row in rows:
    print(row)

(1, 'AYM', 'Dresses', 179.0, 'GBP', '68% Bamboo Viscose, 28% Cotton, 4% Elastane')


# 4. Scrape Website

In [13]:
#get the full list of category URLS

base_url='https://www.aym-studio.com'

nav_links=soup.find_all("a", href=True)
category_urls_raw=[link["href"] for link in nav_links if "/collections/" in link["href"]]

full_urls=[]
for url in category_urls_raw:
    if url.startswith("http"):
        full_urls.append(url)
    else:
        full_urls.append(base_url + url)

full_urls=list(set(full_urls))

## test
print(len(full_urls))
print(full_urls)

28
['https://www.aym-studio.com/collections/styling-pieces', 'https://www.aym-studio.com/collections/occasionwear', 'https://www.aym-studio.com/collections/wedding-wardrobe', 'https://www.aym-studio.com/collections/sculpting-silhouettes', 'https://www.aym-studio.com/collections/all-clothing', 'https://www.aym-studio.com/collections/most-popular', 'https://www.aym-studio.com/collections/maternity', 'https://www.aym-studio.com/collections/confidence-collection', 'https://www.aym-studio.com/collections/multi-way-styles', 'https://www.aym-studio.com/collections/new-arrivals', 'https://www.aym-studio.com/collections/code-collection', 'https://www.aym-studio.com/collections/midi-dress', 'https://www.aym-studio.com/collections/flowing-styles', 'https://www.aym-studio.com/collections/all', 'https://www.aym-studio.com/collections/maxi-dress', 'https://www.aym-studio.com/collections/last-chance-sale', 'https://www.aym-studio.com/collections/day-dresses', 'https://www.aym-studio.com/collections/f

In [14]:
_urls=['https://www.aym-studio.com/collections/all-tops','https://www.aym-studio.com/collections/dresses','https://www.aym-studio.com/collections/all-bottoms']

In [17]:
#get 5 products from each categhttps://www.aym-studio.com/collections/all-topsory 

product_urls=set()

for category_url in relevant_urls:
    print(f"Scraping category: {category_url}")
    
    response=requests.get(category_url)
    soup=BeautifulSoup(response.text, "html.parser")
    
    links=soup.find_all("a", href=True)
    
    category_products=set()
    
    for link in links:
        href=link["href"]
        
        if "/products/" in href:
            if href.startswith("http"):
                full_url=href
            else:
                full_url= base_url+href
            
            category_products.add(full_url)
    
    category_products=list(category_products)[:]
    
    print(f"  Found {len(category_products)} products")
    
    # add to global set
    product_urls.update(category_products)

print(f"\nTotal unique products: {len(product_urls)}")
print(list(product_urls)[:])

Scraping category: https://www.aym-studio.com/collections/all-tops
  Found 16 products
Scraping category: https://www.aym-studio.com/collections/dresses
  Found 28 products
Scraping category: https://www.aym-studio.com/collections/all-bottoms
  Found 15 products

Total unique products: 47
['https://www.aym-studio.com/products/venus-long-sleeve-top', 'https://www.aym-studio.com/products/apartment-trousers', 'https://www.aym-studio.com/products/bamboo-wrap-top?variant=56335833006457&color=earth-red', 'https://www.aym-studio.com/products/selene-top', 'https://www.aym-studio.com/products/finsbury-trousers', 'https://www.aym-studio.com/products/theo-turtleneck-long-sleeve-top-in-organic-bamboo', 'https://www.aym-studio.com/products/styling-scarf?variant=56145037263225&color=wine-red', 'https://www.aym-studio.com/products/venus-dress', 'https://www.aym-studio.com/products/fletcher-skirt', 'https://www.aym-studio.com/products/diana-top', 'https://www.aym-studio.com/products/nova-maxi-dress?va

In [22]:
for url in product_urls:
    print(f"Scraping: {url}")
    
    try:
        response=requests.get(url)
        soup=BeautifulSoup(response.text, "html.parser")
        
        #JSON extraction
        script_tag=soup.find("script", type="application/ld+json")
        json_data=json.loads(script_tag.string)

        brand=json_data["brand"]["name"]      
        category=json_data["category"]        
        product_name=json_data["name"]        
        price=json_data["hasVariant"][0]["offers"]["price"]  
        currency=json_data["hasVariant"][0]["offers"]["priceCurrency"]  
        
        
        #html extraction 
        materials_raw=None
        
        material_blocks=soup.find_all(
            "span",
            class_="metafield-multi_line_text_field"
        )
        
        for block in material_blocks:
            text=block.get_text(separator=" ", strip=True)
            if "%" in text:
                materials_raw=text
                break
        
        if materials_raw:
            materials_clean=materials_raw.split(".")[0]
        else:
            materials_clean=None
        
        #insert into databse
        cursor.execute("""
        INSERT INTO products (brand, category, price, currency, materials)
        VALUES (?, ?, ?, ?, ?)
        """, (brand, category, price, currency, materials_clean))
        
    except Exception as e:
        print(f"Error with {url}: {e}")

conn.commit()

Scraping: https://www.aym-studio.com/products/ava-reversible-maxi-dress
Scraping: https://www.aym-studio.com/products/angelina-maxi-dress?variant=54996242301305
Scraping: https://www.aym-studio.com/products/venus-dress?_pos=2&_fid=2eebd2c9b&_ss=c&variant=54809477120377
Scraping: https://www.aym-studio.com/products/audrey-midi-dress-in-organic-bamboo?_pos=3&_fid=2eebd2c9b&_ss=c&variant=44591258796265
Scraping: https://www.aym-studio.com/products/harper-mini-dress
Scraping: https://www.aym-studio.com/products/angelina-maxi-dress?variant=54996242268537
Scraping: https://www.aym-studio.com/products/audrey-midi-dress-in-organic-bamboo?variant=45687327817961
Scraping: https://www.aym-studio.com/products/penelope-dress
Scraping: https://www.aym-studio.com/products/emmie-dress
Scraping: https://www.aym-studio.com/products/short-drape-cape?variant=56143980986745
Scraping: https://www.aym-studio.com/products/sammi-shorts-in-bamboo
Scraping: https://www.aym-studio.com/products/juni-midi-dress
Scr

In [29]:
#turning data into dataframe to visualize

cursor.execute("SELECT * FROM products")
rows=cursor.fetchall()
df=pd.DataFrame(rows, columns=["id", "brand", "category", "price", "currency", "materials"])

In [33]:
## test

print(df)
print(df['category'].unique())

    id brand          category  price currency  \
0    1   AYM           Dresses  179.0      GBP   
1    2   AYM           Dresses  159.0      GBP   
2    3   AYM           Dresses  189.0      GBP   
3    4   AYM           Dresses  159.0      GBP   
4    5   AYM           Dresses  119.0      GBP   
5    6   AYM           Dresses   89.0      GBP   
6    7   AYM           Dresses  189.0      GBP   
7    8   AYM           Dresses  119.0      GBP   
8    9   AYM           Dresses  159.0      GBP   
9   10   AYM           Dresses  149.0      GBP   
10  11   AYM             Capes   49.0      GBP   
11  12   AYM            Shorts   59.0      GBP   
12  13   AYM           Dresses  129.0      GBP   
13  14   AYM             Capes   49.0      GBP   
14  15   AYM           Dresses  159.0      GBP   
15  16   AYM           Dresses  139.0      GBP   
16  17   AYM        One-Pieces  149.0      GBP   
17  18   AYM           Dresses  159.0      GBP   
18  19   AYM     Clothing Tops   79.0      GBP   


# 5. Conclusion 

This prototyping notebook allowed us to explore the raw data, test assumptions, and identify issues not visible initially. It led to the following conclusions:

* The `category` field is inconsistent and too granular, so it is split into `category_raw` (original value) and `category_clean` (standardized category).
* A `product_id` (from the cleaned URL) is added to uniquely identify products across scraping runs.
* A `scrape_date` field is included to track when each record was collected and enable time-based analysis.
* Materials are split into `materials_raw` and `materials_clean` to preserve original data while enabling easier use.

**Final proposed schema:**

* id
* brand
* product_name
* product_url
* product_id
* category_raw
* category_clean
* price
* currency
* materials_raw
* materials_clean
* scrape_date


In [ ]:
project/
│
├── core/
│   ├── http.py
│   ├── parsers.py
│   ├── extractors.py
│   ├── crawler.py
│
├── scrapers/
│   ├── aym.py
│   ├── zara.py
│
├── config/
│   └── sites.py
│
└── main.py

In [ ]:
def collect_product_urls(base_url, category_paths):
    all_urls = []

    for path in category_paths:
        url = base_url + path
        soup = get_soup(url)

        relative_urls = extract_product_urls(soup)
        full_urls = normalize_links(base_url, relative_urls)

        all_urls.extend(full_urls)

    return list(set(all_urls))

In [18]:
project/
│
├── extract/
│   ├── links.py
│   ├── product.py
│   ├── requests.py
│
├── transform/
│   ├── cleaning.py
│
├── load/
│   ├── insert.py
│
├── utils/
│   ├── helpers.py
│
├── main.py

SyntaxError: invalid character '│' (U+2502) (3864590985.py, line 2)

In [ ]:
project/
│
├── extract/
│   ├── core/           # reusable logic
│   │   ├── http.py
│   │   ├── crawler.py
│   │   ├── parsers.py
│   │
│   ├── sites/          # site-specific logic
│   │   ├── aym.py
│   │   ├── zara.py
│
├── transform/
├── load/
├── main.py

In [ ]:
project/
│
├── extract/
│   ├── __init__.py          ✅ package
│   │
│   ├── core/
│   │   ├── __init__.py      ✅ package
│   │   ├── http.py
│   │   ├── crawler.py
│   │   ├── parsers.py
│   │
│   ├── sites/
│   │   ├── __init__.py      ✅ package
│   │   ├── aym.py
│   │   ├── zara.py
│
├── transform/
│   ├── __init__.py          ✅ package
│   ├── cleaning.py
│
├── load/
│   ├── __init__.py          ✅ package
│   ├── insert.py
│
├── main.py

In [ ]:
project/
│──00_etl_prototype.ipynb
├── extract/
│   ├── __init__.py
│   │
│   ├── shared/
│   │   ├── __init__.py
│   │   ├── http.py          # get_soup, request logic
│   │   ├── links.py         # extract_product_url
│   │   ├── html_extractors.py       # extract_json_block, extract_materials
│   │
│   ├── scraper.py          # 👈 shared scraping workflow (same for all sites)
│   │
│   ├── sites/
│   │   ├── aym/
│   │   │   ├── __init__.py  
│   │   │   ├── config.py     # CONFIG dict (base_url, locators, etc.)
│   │   │   ├── adapter.py   # 👈 ONLY site-specific extraction logic
│   │   │
│   │   ├── indigo_luna/             
│   │   │   ├── __init__.py  
│   │   │   ├── config.py
│   │   │   ├── adapter.py
transform/
│
├── cleaning.py
├── semantic/
│   ├── __init__.py
│   ├── orchestrator.py          # Runs Agent 1 → Agent 2
│   ├── schemas.py               # Pydantic models
│   ├── prompts.py               # Prompt templates
│   ├── tools.py                 # lookup_certification()
│   │
│   ├── agents/
│   │   ├── __init__.py
│   │   ├── composition_agent.py
│   │   └── attribution_agent.py
│   │
│   └── cache.py                 # Optional certification cache
├── load/
│   ├── insert.py             # database insertion logic
│
├── main.py                   # 👈 orchestration (runs the pipeline)

In [ ]:

  {
    "@context": "http://schema.org/",
    "@type": "Product",
    "name": "Elemental Cardi Silver",
    "url": "https:\/\/indigoluna.store\/products\/elemental-cardi-silver",
    "image": [
        "https:\/\/indigoluna.store\/cdn\/shop\/files\/indigoluna-2.jpg?v=1776836858\u0026width=1920"
      ],
    "description": "An essential layering piece made from 100% cotton, the Elemental Cardigan adds warmth, texture, and softness to any outfit. Whether you button it up for a vintage bomber look, or wear it open for a more androgynous, urban style, this piece was made to hold you through all seasons of life.",
    "sku": "ELCARSI-1",
    "brand": {
      "@type": "Brand",
      "name": "Indigo Luna"
    },
    "offers": [{
          "@type" : "Offer","sku": "ELCARSI-1","availability" : "http://schema.org/InStock",
          "price" : 98.0,
          "priceCurrency" : "GBP",
          "url" : "https:\/\/indigoluna.store\/products\/elemental-cardi-silver?variant=42293596454983"
        },
{
          "@type" : "Offer","sku": "ELCARSI-2","availability" : "http://schema.org/InStock",
          "price" : 98.0,
          "priceCurrency" : "GBP",
          "url" : "https:\/\/indigoluna.store\/products\/elemental-cardi-silver?variant=42293596487751"
        }
]
  }
